# RQ2 Context Selection: Preliminary Evaluation

This notebook reconstructs a review-level evidence funnel from the saved RQ2 outputs. It is intentionally diagnostic: it reports available and complete runs separately, merges the C1 and A3 continuation fragments, and does not treat partial C5 or T2 results as final comparisons.

The evaluation separates model-generated learnings from the underlying evidence representation. Branch-context recall measures the NCT IDs represented by selected chunks or trial blocks before global context assembly. Final evidence recall measures the identifiers retained in compressed chunks or deterministic trial blocks after the final budget, while final learning recall measures identifiers mentioned in the retained generated syntheses. Grounded learning recall counts only learning citations that can be traced to the selected branch evidence. Combined final-context recall is the textual union received by the report-writing model, while grounded combined recall excludes untraceable learning citations.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / 'gpt_researcher').is_dir() and (path / 'outputs').is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')

OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'
ANALYSIS_DIR = PROJECT_ROOT / 'outputs' / 'evaluation' / 'context'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_REVIEWS = 51
NCT_PATTERN = re.compile(r'\bNCT\d{8}\b', re.IGNORECASE)

RUN_SPECS = {
    'C1': {
        'label': 'C1 MiniLM, 30 x 1000',
        'fragments': [
            'rq2_c1_minilm_t035_max30_chunk1000',
            'rq2_c1_minilm_t035_max30_chunk1000_resume_26_51',
        ],
        'representation': 'compression', 'dedupe': False, 'rerank': False,
    },
    'C2': {
        'label': 'C2 BGE, 30 x 1000',
        'fragments': ['rq2_c2_bge_t035_max30_chunk1000'],
        'representation': 'compression', 'dedupe': False, 'rerank': False,
    },
    'C3': {
        'label': 'C3 BGE, threshold 0.30',
        'fragments': ['rq2_c3_bge_t030_max30_chunk1000'],
        'representation': 'compression', 'dedupe': False, 'rerank': False,
    },
    'C4': {
        'label': 'C4 BGE, 80 x 1000',
        'fragments': ['rq2_c4_bge_t035_max80_chunk1000'],
        'representation': 'compression', 'dedupe': False, 'rerank': False,
    },
    'C5': {
        'label': 'C5 BGE, 40 x 2000',
        'fragments': ['rq2_c5_bge_t035_max40_chunk2000'],
        'representation': 'compression', 'dedupe': False, 'rerank': False,
    },
    'A2': {
        'label': 'A2 Compression + dedupe',
        'fragments': ['rq2_a2_bge_compression_dedupe_no_rerank'],
        'representation': 'compression', 'dedupe': True, 'rerank': False,
    },
    'A3': {
        'label': 'A3 Compression + dedupe + rerank',
        'fragments': [
            'rq2_a3_bge_compression_dedupe_rerank',
            'rq2_a3_bge_compression_dedupe_rerank_resume_13_51',
        ],
        'representation': 'compression', 'dedupe': True, 'rerank': True,
    },
    'A4': {
        'label': 'A4 Trial blocks, 30 x 1600',
        'fragments': ['rq2_a4_trial_blocks_top30_chars1600'],
        'representation': 'trial_blocks', 'dedupe': True, 'rerank': True,
    },
    'T1': {
        'label': 'T1 Trial blocks, 30 x 1000',
        'fragments': ['rq2_t1_trial_blocks_top30_chars1000'],
        'representation': 'trial_blocks', 'dedupe': True, 'rerank': True,
    },
    'T2': {
        'label': 'T2 Trial blocks, 50 x 1600',
        'fragments': ['rq2_t2_trial_blocks_top50_chars1600'],
        'representation': 'trial_blocks', 'dedupe': True, 'rerank': True,
    },
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Analysis output: {ANALYSIS_DIR.resolve()}')

## 1. Load and validate logical experiments

C1 and A3 are logical experiments assembled from two non-overlapping execution fragments. If overlapping PMIDs are encountered, a successful record is preferred over a failed record.

In [ ]:
def read_json(path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def load_logical_run(run_id, spec):
    merged = {}
    fragment_info = []
    for fragment in spec['fragments']:
        run_dir = OUTPUTS_ROOT / fragment
        evaluation_path = run_dir / 'evaluation_run.json'
        if not evaluation_path.is_file():
            fragment_info.append({'fragment': fragment, 'available': False, 'records': 0})
            continue
        payload = read_json(evaluation_path)
        records = payload.get('queries', [])
        fragment_info.append({'fragment': fragment, 'available': True, 'records': len(records)})
        for source_record in records:
            record = dict(source_record)
            record['_run_id'] = run_id
            record['_fragment'] = fragment
            record['_run_dir'] = str(run_dir)
            pmid = str(record.get('pmid', ''))
            previous = merged.get(pmid)
            if previous is None or (record.get('status') == 'ok' and previous.get('status') != 'ok'):
                merged[pmid] = record
    return merged, fragment_info


LOGICAL_RUNS = {}
fragment_rows = []
validation_rows = []
for run_id, spec in RUN_SPECS.items():
    records, fragment_info = load_logical_run(run_id, spec)
    LOGICAL_RUNS[run_id] = records
    fragment_rows.extend({'run_id': run_id, **row} for row in fragment_info)
    statuses = Counter(record.get('status', 'missing') for record in records.values())
    validation_rows.append({
        'run_id': run_id,
        'label': spec['label'],
        'records': len(records),
        'ok_records': statuses.get('ok', 0),
        'other_statuses': ', '.join(
            f'{status}={count}' for status, count in sorted(statuses.items()) if status != 'ok'
        ),
        'all_reviews_ok': len(records) == EXPECTED_REVIEWS and statuses.get('ok', 0) == EXPECTED_REVIEWS,
    })

fragment_df = pd.DataFrame(fragment_rows)
validation_df = pd.DataFrame(validation_rows)
display(validation_df)
display(fragment_df)

## 2. Reconstruct the evidence funnel

The stages are: candidate-producing retrieval excluding planning search, input to branch representation, reranker input and output where applicable, selected branch context, assembled context before final trimming, and final context after trimming.

In [ ]:
def normalise_nct_ids(values):
    if values is None:
        return set()
    if isinstance(values, dict):
        values = values.keys()
    result = set()
    for value in values:
        match = NCT_PATTERN.fullmatch(str(value).strip())
        if match:
            result.add(match.group(0).upper())
    return result


def resolve_debug_dir(record):
    raw_path = record.get('final_context_debug_dir')
    if raw_path:
        candidate = Path(raw_path)
        if not candidate.is_absolute():
            candidate = PROJECT_ROOT / candidate
        if candidate.is_dir():
            return candidate
    fallback_root = Path(record['_run_dir']) / 'final_context_debug'
    if fallback_root.is_dir():
        pmid = str(record.get('pmid', ''))
        candidates = [path for path in fallback_root.iterdir() if path.is_dir() and f'_{pmid}_' in path.name]
        if candidates:
            return max(candidates, key=lambda path: path.stat().st_mtime)
    return None


def load_first_json(directory, pattern):
    if directory is None:
        return None
    matches = sorted(directory.glob(pattern))
    return read_json(matches[0]) if matches else None


def load_all_json(directory, pattern):
    if directory is None:
        return []
    return [read_json(path) for path in sorted(directory.glob(pattern))]


CONTEXT_ITEM_SEPARATOR = '\n\n---\n\n'


def split_context_components(text, representation):
    """Split packed context into generated learnings and retained evidence."""
    items = [
        item.strip()
        for item in (text or '').split(CONTEXT_ITEM_SEPARATOR)
        if item.strip()
    ]
    if representation == 'trial_blocks':
        is_evidence = lambda item: bool(re.search(r'(?m)^### Trial\s+\d+:', item))
    else:
        is_evidence = lambda item: bool(re.match(r'^\s*Source:\s*', item, flags=re.IGNORECASE))

    first_evidence = next(
        (index for index, item in enumerate(items) if is_evidence(item)),
        len(items),
    )
    learning_items = items[:first_evidence]
    evidence_items = items[first_evidence:]
    parse_is_ordered = all(not is_evidence(item) for item in learning_items) and all(
        is_evidence(item) for item in evidence_items
    )
    return learning_items, evidence_items, parse_is_ordered


def ids_from_items(items):
    return normalise_nct_ids(NCT_PATTERN.findall(CONTEXT_ITEM_SEPARATOR.join(items)))


def words_in_items(items):
    return len(CONTEXT_ITEM_SEPARATOR.join(items).split())


def item_word_total(items):
    """Count item words as the runtime packer does, excluding separators."""
    return sum(len(item.split()) for item in items)


def split_compressed_branch_chunks(evidence_items):
    """Recover the individual Source/Title/Content chunks inside each branch string."""
    branches = []
    parse_ok = True
    for branch_text in evidence_items:
        chunks = [
            chunk.strip()
            for chunk in re.split(r'(?mi)(?=^[ \t]*Source:\s*)', branch_text)
            if chunk.strip()
        ]
        parse_ok = parse_ok and bool(chunks) and all(
            re.match(r'^\s*Source:\s*', chunk, flags=re.IGNORECASE) for chunk in chunks
        )
        branches.append(chunks)
    return branches, parse_ok


def interleave_ranked_branches(branches):
    """Round-robin branch items while preserving rank within every branch."""
    if not branches:
        return []
    ordered = []
    for rank in range(max((len(branch) for branch in branches), default=0)):
        for branch in branches:
            if rank < len(branch):
                ordered.append(branch[rank])
    return ordered


def pack_items_from_start(items, word_budget):
    """Mirror the runtime packer: retain fitting items and skip oversized ones."""
    retained = []
    used_words = 0
    for item in items:
        item_words = len(item.split())
        if used_words + item_words > word_budget:
            continue
        retained.append(item)
        used_words += item_words
    return retained, used_words


def resolved_config(record):
    config_path = Path(record['_run_dir']) / 'resolved_config.env'
    config = {}
    if config_path.is_file():
        for raw_line in config_path.read_text(encoding='utf-8', errors='replace').splitlines():
            line = raw_line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            config[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
    return config


def recall(ids, ground_truth):
    return len(ids & ground_truth) / len(ground_truth) if ground_truth else np.nan


def safe_retention(later_ids, earlier_ids, ground_truth):
    earlier_matches = len(earlier_ids & ground_truth)
    return len(later_ids & ground_truth) / earlier_matches if earlier_matches else np.nan


def reconstruct_review(run_id, record):
    spec = RUN_SPECS[run_id]
    gt = normalise_nct_ids(record.get('ground_truth_nct_ids'))
    retrieved = normalise_nct_ids(record.get('retrieved_actual_nct_ids'))
    debug_dir = resolve_debug_dir(record)
    audit = load_first_json(debug_dir, '*_final_context_debug.json')
    web_meta = load_all_json(debug_dir / 'web_context' if debug_dir else None, '*_meta.json')
    rerank_meta = load_all_json(debug_dir / 'llm_nct_rerank' if debug_dir else None, '*.json')

    representation_input = set()
    branch_context = set()
    for metadata in web_meta:
        representation_input |= normalise_nct_ids(metadata.get('scraped_candidate_nct_ids'))
        branch_context |= normalise_nct_ids(metadata.get('web_context_nct_ids'))

    reranker_input = None
    reranker_output = None
    if rerank_meta:
        reranker_input = set()
        reranker_output = set()
        for metadata in rerank_meta:
            reranker_input |= normalise_nct_ids(metadata.get('retrieval_rrf'))
            if not metadata.get('retrieval_rrf'):
                reranker_input |= normalise_nct_ids(metadata.get('llm_scores'))
            reranker_output |= normalise_nct_ids(metadata.get('selected_nct_ids'))

    pre_trim = normalise_nct_ids(audit.get('pre_trim_nct_ids')) if audit else set()
    final_context = normalise_nct_ids(audit.get('post_trim_nct_ids')) if audit else set()

    pre_context_text = ''
    post_context_text = ''
    if debug_dir:
        pre_files = sorted(debug_dir.glob('*_pre_trim_context.txt'))
        if pre_files:
            pre_context_text = pre_files[0].read_text(encoding='utf-8', errors='replace')
        post_files = sorted(debug_dir.glob('*_post_trim_context.txt'))
        if post_files:
            post_context_text = post_files[0].read_text(encoding='utf-8', errors='replace')

    pre_learning_items, pre_evidence_items, pre_parse_ok = split_context_components(
        pre_context_text, spec['representation']
    )
    final_learning_items, final_evidence_items, final_parse_ok = split_context_components(
        post_context_text, spec['representation']
    )
    pre_learning_context = ids_from_items(pre_learning_items)
    pre_evidence_context = ids_from_items(pre_evidence_items)
    final_learning_context = ids_from_items(final_learning_items)
    final_evidence_context = ids_from_items(final_evidence_items)
    parsed_final_context = final_learning_context | final_evidence_context
    grounded_final_learning_context = final_learning_context & branch_context
    grounded_final_context = final_evidence_context | grounded_final_learning_context
    nct_mentions = NCT_PATTERN.findall(post_context_text)

    config = resolved_config(record)
    try:
        final_context_max_words = int(config.get('FINAL_CONTEXT_MAX_WORDS', 9000))
    except (TypeError, ValueError):
        final_context_max_words = 9000

    counterfactual_chunks = []
    counterfactual_evidence_context = None
    counterfactual_grounded_final_context = None
    counterfactual_chunk_parse_ok = np.nan
    counterfactual_available_chunks = np.nan
    counterfactual_evidence_words = np.nan
    counterfactual_trial_budget_words = np.nan
    if spec['representation'] == 'compression' and audit is not None:
        branch_chunks, counterfactual_chunk_parse_ok = split_compressed_branch_chunks(
            pre_evidence_items
        )
        counterfactual_chunk_parse_ok = bool(pre_evidence_items) and counterfactual_chunk_parse_ok
        ordered_chunks = interleave_ranked_branches(branch_chunks)
        counterfactual_trial_budget_words = max(
            0, final_context_max_words - item_word_total(final_learning_items)
        )
        counterfactual_chunks, counterfactual_evidence_words = pack_items_from_start(
            ordered_chunks, counterfactual_trial_budget_words
        )
        counterfactual_available_chunks = len(ordered_chunks)
        counterfactual_evidence_context = ids_from_items(counterfactual_chunks)
        counterfactual_grounded_final_context = (
            counterfactual_evidence_context | grounded_final_learning_context
        )

    row = {
        'run_id': run_id,
        'label': spec['label'],
        'representation': spec['representation'],
        'dedupe': spec['dedupe'],
        'rerank': spec['rerank'],
        'pmid': str(record.get('pmid', '')),
        'title': record.get('title', ''),
        'status': record.get('status'),
        'fragment': record.get('_fragment'),
        'gt_count': len(gt),
        'debug_available': audit is not None,
        'web_context_files': len(web_meta),
        'reranker_files': len(rerank_meta),
        'pre_trim_items': audit.get('pre_trim_item_count', np.nan) if audit else np.nan,
        'final_items': audit.get('post_trim_item_count', np.nan) if audit else np.nan,
        'pre_trim_words': audit.get('pre_trim_word_count', np.nan) if audit else np.nan,
        'final_words': audit.get('post_trim_word_count', np.nan) if audit else np.nan,
        'pre_learning_items': len(pre_learning_items),
        'pre_evidence_items': len(pre_evidence_items),
        'final_learning_items': len(final_learning_items),
        'final_evidence_items': len(final_evidence_items),
        'pre_learning_words': words_in_items(pre_learning_items),
        'pre_evidence_words': words_in_items(pre_evidence_items),
        'final_learning_words': words_in_items(final_learning_items),
        'final_evidence_words': words_in_items(final_evidence_items),
        'final_context_max_words': final_context_max_words,
        'counterfactual_chunk_parse_ok': counterfactual_chunk_parse_ok,
        'counterfactual_available_chunks': counterfactual_available_chunks,
        'counterfactual_packed_chunks': (
            len(counterfactual_chunks) if spec['representation'] == 'compression' else np.nan
        ),
        'counterfactual_trial_budget_words': counterfactual_trial_budget_words,
        'counterfactual_evidence_words': counterfactual_evidence_words,
        'component_parse_ok': pre_parse_ok and final_parse_ok,
        'component_union_matches_audit': parsed_final_context == final_context,
        'final_nct_mentions': len(nct_mentions),
        'final_mentions_per_unique_nct': len(nct_mentions) / len(final_context) if final_context else np.nan,
    }

    stages = {
        'retrieval': retrieved,
        'representation_input': representation_input,
        'reranker_input': reranker_input,
        'reranker_output': reranker_output,
        'branch_context': branch_context,
        'pre_trim_learning_context': pre_learning_context,
        'pre_trim_evidence_context': pre_evidence_context,
        'pre_trim_context': pre_trim,
        'final_learning_context': final_learning_context,
        'grounded_final_learning_context': grounded_final_learning_context,
        'final_evidence_context': final_evidence_context,
        'grounded_final_context': grounded_final_context,
        'final_context': final_context,
        'counterfactual_evidence_context': counterfactual_evidence_context,
        'counterfactual_grounded_final_context': counterfactual_grounded_final_context,
    }
    for stage, ids in stages.items():
        if ids is None:
            row[f'{stage}_unique'] = np.nan
            row[f'{stage}_matched'] = np.nan
            row[f'{stage}_recall'] = np.nan
        else:
            row[f'{stage}_unique'] = len(ids)
            row[f'{stage}_matched'] = len(ids & gt)
            row[f'{stage}_recall'] = recall(ids, gt)

    row['retrieval_to_final_gt_retention'] = safe_retention(final_context, retrieved, gt)
    row['branch_to_final_gt_retention'] = safe_retention(final_context, branch_context, gt)
    row['pre_to_post_gt_retention'] = safe_retention(final_context, pre_trim, gt)
    row['evidence_pre_to_final_gt_retention'] = safe_retention(
        final_evidence_context, pre_evidence_context, gt
    )
    row['learning_pre_to_final_gt_retention'] = safe_retention(
        final_learning_context, pre_learning_context, gt
    )
    learning_only = final_learning_context - final_evidence_context
    evidence_only = final_evidence_context - final_learning_context
    component_overlap = final_learning_context & final_evidence_context
    row['final_learning_only_gt_matched'] = len(learning_only & gt)
    row['final_evidence_only_gt_matched'] = len(evidence_only & gt)
    row['final_component_overlap_gt_matched'] = len(component_overlap & gt)
    row['final_learning_only_gt_recall'] = recall(learning_only, gt)
    row['final_evidence_only_gt_recall'] = recall(evidence_only, gt)
    row['final_component_overlap_gt_recall'] = recall(component_overlap, gt)
    available_learning_ncts = grounded_final_learning_context
    missing_learning_ncts = final_learning_context - branch_context
    row['learning_citation_available_count'] = len(available_learning_ncts)
    row['learning_citation_missing_count'] = len(missing_learning_ncts)
    row['learning_citation_missing_gt_count'] = len(missing_learning_ncts & gt)
    row['learning_citation_availability'] = (
        len(available_learning_ncts) / len(final_learning_context)
        if final_learning_context else np.nan
    )
    row['learning_citation_missing_nct_ids'] = missing_learning_ncts
    if reranker_input is not None and reranker_output is not None:
        row['reranker_gt_retention'] = safe_retention(reranker_output, reranker_input, gt)
    else:
        row['reranker_gt_retention'] = np.nan
    row['final_gt_per_1000_words'] = (
        1000 * row['final_context_matched'] / row['final_words']
        if row['final_words'] and not pd.isna(row['final_words']) else np.nan
    )
    row['counterfactual_evidence_gt_per_1000_words'] = (
        1000 * row['counterfactual_evidence_context_matched']
        / row['counterfactual_evidence_words']
        if row['counterfactual_evidence_words']
        and not pd.isna(row['counterfactual_evidence_words']) else np.nan
    )
    return row


In [ ]:
review_rows = []
for run_id, records in LOGICAL_RUNS.items():
    for record in records.values():
        review_rows.append(reconstruct_review(run_id, record))

reviews_df = pd.DataFrame(review_rows)
reviews_df.to_csv(ANALYSIS_DIR / 'per_review_context_funnel.csv', index=False)

debug_counts = (
    reviews_df.groupby('run_id', as_index=False)
    .agg(
        debug_records=('debug_available', 'sum'),
        component_parse_failures=('component_parse_ok', lambda values: int((~values).sum())),
        component_audit_mismatches=(
            'component_union_matches_audit', lambda values: int((~values).sum())
        ),
        counterfactual_chunk_parse_failures=(
            'counterfactual_chunk_parse_ok',
            lambda values: int((~values.dropna().astype(bool)).sum()),
        ),
    )
)
validation_df = validation_df.merge(debug_counts, on='run_id', how='left')
validation_df['debug_records'] = validation_df['debug_records'].fillna(0).astype(int)
validation_df['complete_for_context_analysis'] = (
    validation_df['all_reviews_ok']
    & (validation_df['debug_records'] == EXPECTED_REVIEWS)
    & (validation_df['component_parse_failures'] == 0)
    & (validation_df['component_audit_mismatches'] == 0)
)
validation_df.to_csv(ANALYSIS_DIR / 'run_validation.csv', index=False)
display(validation_df)

## 3. Preliminary result tables

The available-results table is useful while queues are running. The complete-results table should be used for reporting. Failed reviews and reviews without a final-context audit are excluded from metric averages and remain visible in the validation table.

In [ ]:
STAGES = [
    'retrieval', 'representation_input', 'reranker_input', 'reranker_output',
    'branch_context', 'pre_trim_learning_context', 'pre_trim_evidence_context',
    'pre_trim_context', 'final_learning_context', 'grounded_final_learning_context',
    'final_evidence_context', 'grounded_final_context', 'final_context',
]


def summarise(frame):
    rows = []
    for run_id, group in frame.groupby('run_id', sort=False):
        row = {
            'run_id': run_id,
            'label': RUN_SPECS[run_id]['label'],
            'reviews': len(group),
            'total_gt': int(group['gt_count'].sum()),
            'complete_final_recall_reviews': int((group['final_context_recall'] == 1.0).sum()),
            'zero_final_recall_reviews': int((group['final_context_recall'] == 0.0).sum()),
            'macro_retrieval_to_final_retention': group['retrieval_to_final_gt_retention'].mean(),
            'macro_branch_to_final_retention': group['branch_to_final_gt_retention'].mean(),
            'macro_pre_to_post_retention': group['pre_to_post_gt_retention'].mean(),
            'macro_evidence_pre_to_final_retention': group['evidence_pre_to_final_gt_retention'].mean(),
            'macro_learning_pre_to_final_retention': group['learning_pre_to_final_gt_retention'].mean(),
            'macro_reranker_retention': group['reranker_gt_retention'].mean(),
            'final_learning_only_gt_pairs': int(group['final_learning_only_gt_matched'].sum()),
            'final_evidence_only_gt_pairs': int(group['final_evidence_only_gt_matched'].sum()),
            'final_component_overlap_gt_pairs': int(group['final_component_overlap_gt_matched'].sum()),
            'macro_final_learning_only_gt_recall': group['final_learning_only_gt_recall'].mean(),
            'macro_final_evidence_only_gt_recall': group['final_evidence_only_gt_recall'].mean(),
            'macro_final_component_overlap_gt_recall': group['final_component_overlap_gt_recall'].mean(),
            'mean_learning_citation_availability': group['learning_citation_availability'].mean(),
            'learning_citation_missing_ncts': int(group['learning_citation_missing_count'].sum()),
            'learning_citation_missing_gt_pairs': int(group['learning_citation_missing_gt_count'].sum()),
            'component_parse_failures': int((~group['component_parse_ok']).sum()),
            'component_audit_mismatches': int((~group['component_union_matches_audit']).sum()),
            'mean_final_unique_ncts': group['final_context_unique'].mean(),
            'mean_final_learning_words': group['final_learning_words'].mean(),
            'mean_final_evidence_words': group['final_evidence_words'].mean(),
            'mean_final_words': group['final_words'].mean(),
            'mean_final_gt_per_1000_words': group['final_gt_per_1000_words'].mean(),
            'mean_final_mentions_per_unique_nct': group['final_mentions_per_unique_nct'].mean(),
        }
        for stage in STAGES:
            recall_column = f'{stage}_recall'
            matched_column = f'{stage}_matched'
            row[f'macro_{stage}_recall'] = group[recall_column].mean()
            row[f'{stage}_gt_pairs'] = (
                int(group[matched_column].sum()) if group[matched_column].notna().any() else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)


eligible_df = reviews_df[
    reviews_df['status'].eq('ok') & reviews_df['debug_available']
].copy()
summary_available_df = summarise(eligible_df)
summary_available_df.to_csv(ANALYSIS_DIR / 'summary_available_reviews.csv', index=False)

complete_run_ids = validation_df.loc[
    validation_df['complete_for_context_analysis'], 'run_id'
].tolist()
pmids_by_complete_run = {
    run_id: set(eligible_df.loc[eligible_df['run_id'] == run_id, 'pmid'])
    for run_id in complete_run_ids
}
common_pmids = (
    set.intersection(*pmids_by_complete_run.values()) if pmids_by_complete_run else set()
)
complete_common_df = eligible_df[
    eligible_df['run_id'].isin(complete_run_ids) & eligible_df['pmid'].isin(common_pmids)
].copy()
summary_complete_df = summarise(complete_common_df) if not complete_common_df.empty else pd.DataFrame()
summary_complete_df.to_csv(ANALYSIS_DIR / 'summary_complete_common_reviews.csv', index=False)

PREVIEW_COLUMNS = [
    'run_id', 'reviews', 'macro_retrieval_recall',
    'macro_branch_context_recall', 'macro_final_evidence_context_recall',
    'final_evidence_context_gt_pairs', 'macro_final_learning_context_recall',
    'final_learning_context_gt_pairs', 'macro_grounded_final_learning_context_recall',
    'grounded_final_learning_context_gt_pairs', 'macro_grounded_final_context_recall',
    'grounded_final_context_gt_pairs', 'macro_final_context_recall',
    'final_context_gt_pairs', 'learning_citation_missing_gt_pairs', 'final_learning_only_gt_pairs',
    'final_evidence_only_gt_pairs', 'final_component_overlap_gt_pairs',
    'mean_learning_citation_availability',
    'macro_retrieval_to_final_retention', 'mean_final_unique_ncts',
    'mean_final_learning_words', 'mean_final_evidence_words',
    'mean_final_words', 'mean_final_gt_per_1000_words',
]
print('Available successful reviews, including partial runs:')
display(summary_available_df[PREVIEW_COLUMNS].round(4))
print(f'Complete configurations: {complete_run_ids}')
print(f'Common reviews across complete configurations: {len(common_pmids)}')
if not summary_complete_df.empty:
    display(summary_complete_df[PREVIEW_COLUMNS].round(4))

## 4. Paired configuration comparisons

Each comparison uses only PMIDs successfully audited in both configurations. Rows with fewer than 51 paired reviews are marked as preliminary. The paired bootstrap resamples review-level differences and does not estimate variability across repeated LLM runs.

In [ ]:
PLANNED_COMPARISONS = [
    ('C1', 'C2', 'Embedding model'),
    ('C2', 'C3', 'Similarity threshold'),
    ('C2', 'C4', 'Maximum selected chunks'),
    ('C4', 'C5', 'Chunk granularity'),
    ('C1', 'C4', 'Compression endpoints'),
    ('C4', 'A2', 'NCT consolidation and deduplication'),
    ('A2', 'A3', 'LLM reranking'),
    ('A3', 'A4', 'Compression versus trial blocks'),
    ('A4', 'T1', 'Characters per trial block'),
    ('A4', 'T2', 'Trial-block top K'),
]


def paired_bootstrap(frame, left, right, metric, n_boot=10000, seed=42):
    subset = frame[frame['run_id'].isin([left, right])]
    pivot = subset.pivot(index='pmid', columns='run_id', values=metric)
    if left not in pivot or right not in pivot:
        return None
    pivot = pivot[[left, right]].dropna()
    if pivot.empty:
        return None
    differences = (pivot[right] - pivot[left]).to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(differences), size=(n_boot, len(differences)))
    bootstrap_means = differences[indices].mean(axis=1)
    return {
        'n_paired_reviews': len(differences),
        'left_mean': pivot[left].mean(),
        'right_mean': pivot[right].mean(),
        'mean_difference_right_minus_left': differences.mean(),
        'ci_95_low': np.quantile(bootstrap_means, 0.025),
        'ci_95_high': np.quantile(bootstrap_means, 0.975),
        'improved': int((differences > 1e-12).sum()),
        'tied': int((np.abs(differences) <= 1e-12).sum()),
        'worsened': int((differences < -1e-12).sum()),
        'preliminary_partial_sample': len(differences) < EXPECTED_REVIEWS,
    }


comparison_rows = []
for left, right, comparison_name in PLANNED_COMPARISONS:
    for metric in (
        'branch_context_recall',
        'final_evidence_context_recall',
        'final_learning_context_recall',
        'grounded_final_context_recall',
        'final_context_recall',
    ):
        result = paired_bootstrap(eligible_df, left, right, metric)
        if result:
            result.update({
                'comparison': comparison_name, 'left': left, 'right': right, 'metric': metric,
            })
            comparison_rows.append(result)
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(ANALYSIS_DIR / 'paired_context_comparisons.csv', index=False)
display(comparison_df.round(4))

## 5. Within-run reranker ordering analysis

The stage-level comparison between A2 and A3 also changes the complete pipeline run, so it does not isolate the ordering produced by the reranker. This diagnostic instead uses the saved A3 traces and holds each branch's reranker input fixed. The pre-reranking baseline is the exact stratified order in which the candidates were supplied to the LLM, while the post-reranking order is the final list sorted by LLM score and retrieval tie-breakers.

For each cut-off, the selected candidates are combined across the branches of the same review before recall is calculated. Overall recall uses the complete ground-truth set as its denominator. Candidate-conditioned recall uses only ground-truth trials that appeared in at least one reranker input, separating ordering quality from losses that occurred before reranking. The analysis measures prioritisation of known relevant trials and cannot estimate precision because most non-ground-truth candidates have no relevance judgement.


In [ ]:
RERANKER_K_VALUES = [10, 30, 50, 100]


def ordered_nct_ids(values):
    ordered = []
    seen = set()
    for value in values or []:
        match = NCT_PATTERN.fullmatch(str(value).strip())
        if not match:
            continue
        nct_id = match.group(0).upper()
        if nct_id not in seen:
            ordered.append(nct_id)
            seen.add(nct_id)
    return ordered


def trace_input_order(metadata):
    # retrieval_rrf is serialised from the stratified llm_candidates list,
    # preserving the exact order supplied to the reranker.
    return ordered_nct_ids((metadata.get('retrieval_rrf') or {}).keys())


def union_at_k(branch_rankings, position, k):
    result = set()
    for before, after in branch_rankings:
        ranking = before if position == 'before' else after
        result.update(ranking[:k])
    return result


reranker_review_rows = []
reranker_pair_rows = []
reranker_branch_rows = []
reranker_validation_issues = []

for pmid, record in sorted(LOGICAL_RUNS.get('A3', {}).items()):
    if record.get('status') != 'ok':
        continue

    gt = normalise_nct_ids(record.get('ground_truth_nct_ids'))
    debug_dir = resolve_debug_dir(record)
    trace_paths = sorted(
        (debug_dir / 'llm_nct_rerank').glob('*.json')
    ) if debug_dir else []
    branch_rankings = []
    input_union = set()

    if not trace_paths:
        reranker_validation_issues.append({
            'pmid': pmid,
            'issue': 'missing_reranker_traces',
        })
        continue

    for branch_index, trace_path in enumerate(trace_paths, start=1):
        metadata = read_json(trace_path)
        before = trace_input_order(metadata)
        after = ordered_nct_ids(metadata.get('selected_nct_ids'))
        input_ids = set(before)
        input_union.update(input_ids)
        branch_rankings.append((before, after))

        expected_input = int(metadata.get('input_candidate_count') or 0)
        if len(before) != expected_input:
            reranker_validation_issues.append({
                'pmid': pmid,
                'branch': branch_index,
                'issue': 'input_count_mismatch',
                'expected': expected_input,
                'observed': len(before),
            })
        if not set(after).issubset(input_ids):
            reranker_validation_issues.append({
                'pmid': pmid,
                'branch': branch_index,
                'issue': 'output_not_subset_of_input',
            })

        branch_row = {
            'pmid': pmid,
            'branch': branch_index,
            'input_candidates': len(before),
            'output_candidates': len(after),
            'input_gt': len(input_ids & gt),
        }
        for k in RERANKER_K_VALUES:
            branch_row[f'before_gt_at_{k}'] = len(set(before[:k]) & gt)
            branch_row[f'after_gt_at_{k}'] = len(set(after[:k]) & gt)
        reranker_branch_rows.append(branch_row)

    available_gt = input_union & gt
    review_row = {
        'pmid': pmid,
        'gt_count': len(gt),
        'branches': len(branch_rankings),
        'reranker_input_unique': len(input_union),
        'reranker_input_gt': len(available_gt),
        'reranker_input_recall': recall(input_union, gt),
    }

    for k in RERANKER_K_VALUES:
        before_union = union_at_k(branch_rankings, 'before', k)
        after_union = union_at_k(branch_rankings, 'after', k)
        review_row[f'before_gt_at_{k}'] = len(before_union & gt)
        review_row[f'after_gt_at_{k}'] = len(after_union & gt)
        review_row[f'before_recall_at_{k}'] = recall(before_union, gt)
        review_row[f'after_recall_at_{k}'] = recall(after_union, gt)
        review_row[f'before_conditional_recall_at_{k}'] = (
            len(before_union & available_gt) / len(available_gt)
            if available_gt else np.nan
        )
        review_row[f'after_conditional_recall_at_{k}'] = (
            len(after_union & available_gt) / len(available_gt)
            if available_gt else np.nan
        )

    for nct_id in sorted(available_gt):
        before_ranks = [
            before.index(nct_id) + 1
            for before, _ in branch_rankings
            if nct_id in before
        ]
        after_ranks = [
            after.index(nct_id) + 1
            for _, after in branch_rankings
            if nct_id in after
        ]
        best_before = min(before_ranks)
        best_after = min(after_ranks) if after_ranks else np.inf
        if best_before > 30 and best_after <= 30:
            top30_change = 'Promoted into top 30'
        elif best_before <= 30 and best_after > 30:
            top30_change = 'Demoted from top 30'
        elif best_before <= 30 and best_after <= 30:
            top30_change = 'Retained in top 30'
        else:
            top30_change = 'Outside top 30 in both'
        reranker_pair_rows.append({
            'pmid': pmid,
            'nct_id': nct_id,
            'best_rank_before': best_before,
            'best_rank_after': best_after,
            'top30_change': top30_change,
        })

    reranker_review_rows.append(review_row)

reranker_review_df = pd.DataFrame(reranker_review_rows)
reranker_pair_df = pd.DataFrame(reranker_pair_rows)
reranker_branch_df = pd.DataFrame(reranker_branch_rows)
reranker_validation_df = pd.DataFrame(reranker_validation_issues)

reranker_summary_rows = []
for k in RERANKER_K_VALUES:
    before = reranker_review_df[f'before_recall_at_{k}']
    after = reranker_review_df[f'after_recall_at_{k}']
    differences = (after - before).dropna().to_numpy(dtype=float)
    rng = np.random.default_rng(42 + k)
    bootstrap_means = differences[
        rng.integers(0, len(differences), size=(10000, len(differences)))
    ].mean(axis=1)
    reranker_summary_rows.append({
        'K per branch': k,
        'Retrieval-order macro recall': before.mean(),
        'LLM-reranked macro recall': after.mean(),
        'Mean paired difference': differences.mean(),
        'CI 95% low': np.quantile(bootstrap_means, 0.025),
        'CI 95% high': np.quantile(bootstrap_means, 0.975),
        'Retrieval-order pooled GT pairs': int(
            reranker_review_df[f'before_gt_at_{k}'].sum()
        ),
        'LLM-reranked pooled GT pairs': int(
            reranker_review_df[f'after_gt_at_{k}'].sum()
        ),
        'Retrieval-order conditional recall': reranker_review_df[
            f'before_conditional_recall_at_{k}'
        ].mean(),
        'LLM-reranked conditional recall': reranker_review_df[
            f'after_conditional_recall_at_{k}'
        ].mean(),
        'Reviews improved': int((differences > 1e-12).sum()),
        'Reviews unchanged': int((np.abs(differences) <= 1e-12).sum()),
        'Reviews worsened': int((differences < -1e-12).sum()),
    })

reranker_ordering_summary_df = pd.DataFrame(reranker_summary_rows)
top30_transition_df = (
    reranker_pair_df['top30_change']
    .value_counts()
    .reindex([
        'Retained in top 30',
        'Promoted into top 30',
        'Demoted from top 30',
        'Outside top 30 in both',
    ], fill_value=0)
    .rename_axis('Ground-truth pair transition')
    .reset_index(name='Pairs')
)

reranker_ordering_summary_df.to_csv(
    ANALYSIS_DIR / 'a3_reranker_ordering_summary.csv', index=False
)
reranker_review_df.to_csv(
    ANALYSIS_DIR / 'a3_reranker_ordering_per_review.csv', index=False
)
reranker_pair_df.to_csv(
    ANALYSIS_DIR / 'a3_reranker_top30_pair_transitions.csv', index=False
)
reranker_branch_df.to_csv(
    ANALYSIS_DIR / 'a3_reranker_ordering_per_branch.csv', index=False
)

print(f'A3 reviews analysed: {len(reranker_review_df)}')
print(f'Reranker branch traces analysed: {len(reranker_branch_df)}')
print(f'Known relevant pairs entering at least one reranker: {len(reranker_pair_df)}')
print(f'Validation issues: {len(reranker_validation_df)}')
display(reranker_ordering_summary_df.round(4))
display(top30_transition_df)


## 6. Counterfactual chunk-level final packing

The production pipeline treats each compressed branch string as one indivisible evidence item during final context packing. This diagnostic instead recovers the individual `Source`/`Title`/`Content` chunks from every pre-trim branch, interleaves equal-ranked chunks across branches, and packs them within the words left after the learnings retained by the real run. It does not repeat retrieval, compression, reranking, or learning generation. The result is therefore a counterfactual estimate of the evidence that could have survived if only the final packing unit had been changed from a complete branch string to an individual compressed chunk. Repeated chunks from the same NCT are deliberately retained because they are part of the compression representation.

In [ ]:
counterfactual_df = eligible_df[
    eligible_df['representation'].eq('compression')
    & eligible_df['counterfactual_chunk_parse_ok'].eq(True)
].copy()

counterfactual_rows = []
for run_id, group in counterfactual_df.groupby('run_id', sort=False):
    available = group['counterfactual_available_chunks'].replace(0, np.nan)
    counterfactual_rows.append({
        'run_id': run_id,
        'label': RUN_SPECS[run_id]['label'],
        'reviews': len(group),
        'total_gt': int(group['gt_count'].sum()),
        'actual_evidence_macro_recall': group['final_evidence_context_recall'].mean(),
        'chunk_packed_evidence_macro_recall': group['counterfactual_evidence_context_recall'].mean(),
        'evidence_macro_recall_gain': (
            group['counterfactual_evidence_context_recall']
            - group['final_evidence_context_recall']
        ).mean(),
        'actual_evidence_gt_pairs': int(group['final_evidence_context_matched'].sum()),
        'chunk_packed_evidence_gt_pairs': int(
            group['counterfactual_evidence_context_matched'].sum()
        ),
        'actual_grounded_combined_macro_recall': group['grounded_final_context_recall'].mean(),
        'chunk_packed_grounded_combined_macro_recall': (
            group['counterfactual_grounded_final_context_recall'].mean()
        ),
        'grounded_combined_macro_recall_gain': (
            group['counterfactual_grounded_final_context_recall']
            - group['grounded_final_context_recall']
        ).mean(),
        'actual_grounded_combined_gt_pairs': int(group['grounded_final_context_matched'].sum()),
        'chunk_packed_grounded_combined_gt_pairs': int(
            group['counterfactual_grounded_final_context_matched'].sum()
        ),
        'mean_available_chunks': group['counterfactual_available_chunks'].mean(),
        'mean_packed_chunks': group['counterfactual_packed_chunks'].mean(),
        'mean_packed_fraction': (group['counterfactual_packed_chunks'] / available).mean(),
        'mean_actual_evidence_items': group['final_evidence_items'].mean(),
        'mean_actual_evidence_words': group['final_evidence_words'].mean(),
        'mean_chunk_packed_evidence_words': group['counterfactual_evidence_words'].mean(),
        'mean_chunk_packed_unique_ncts': group['counterfactual_evidence_context_unique'].mean(),
        'mean_chunk_packed_gt_per_1000_words': (
            group['counterfactual_evidence_gt_per_1000_words'].mean()
        ),
        'preliminary_partial_sample': len(group) < EXPECTED_REVIEWS,
    })
counterfactual_summary_df = pd.DataFrame(counterfactual_rows)
counterfactual_summary_df.to_csv(
    ANALYSIS_DIR / 'counterfactual_chunk_repacking_summary.csv', index=False
)


def within_run_bootstrap(frame, before, after, n_boot=10000, seed=42):
    paired = frame[[before, after]].dropna()
    if paired.empty:
        return None
    differences = (paired[after] - paired[before]).to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(differences), size=(n_boot, len(differences)))
    bootstrap_means = differences[indices].mean(axis=1)
    return {
        'n_reviews': len(differences),
        'actual_mean': paired[before].mean(),
        'chunk_packed_mean': paired[after].mean(),
        'mean_gain': differences.mean(),
        'ci_95_low': np.quantile(bootstrap_means, 0.025),
        'ci_95_high': np.quantile(bootstrap_means, 0.975),
        'improved': int((differences > 1e-12).sum()),
        'tied': int((np.abs(differences) <= 1e-12).sum()),
        'worsened': int((differences < -1e-12).sum()),
        'preliminary_partial_sample': len(differences) < EXPECTED_REVIEWS,
    }


counterfactual_bootstrap_rows = []
for run_id, group in counterfactual_df.groupby('run_id', sort=False):
    for label, before, after in (
        (
            'Evidence representation',
            'final_evidence_context_recall',
            'counterfactual_evidence_context_recall',
        ),
        (
            'Grounded combined context',
            'grounded_final_context_recall',
            'counterfactual_grounded_final_context_recall',
        ),
    ):
        result = within_run_bootstrap(group, before, after)
        if result:
            result.update({'run_id': run_id, 'metric': label})
            counterfactual_bootstrap_rows.append(result)
counterfactual_bootstrap_df = pd.DataFrame(counterfactual_bootstrap_rows)
counterfactual_bootstrap_df.to_csv(
    ANALYSIS_DIR / 'counterfactual_chunk_repacking_bootstrap.csv', index=False
)


def cross_configuration_bootstrap(
    frame, left_run, right_run, left_metric, right_metric, n_boot=10000, seed=42
):
    left = frame.loc[frame['run_id'] == left_run, ['pmid', left_metric]].rename(
        columns={left_metric: 'left_value'}
    )
    right = frame.loc[frame['run_id'] == right_run, ['pmid', right_metric]].rename(
        columns={right_metric: 'right_value'}
    )
    paired = left.merge(right, on='pmid', how='inner').dropna()
    if paired.empty:
        return None
    differences = (paired['right_value'] - paired['left_value']).to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(differences), size=(n_boot, len(differences)))
    bootstrap_means = differences[indices].mean(axis=1)
    return {
        'left_run': left_run,
        'right_run': right_run,
        'n_paired_reviews': len(differences),
        'left_mean': paired['left_value'].mean(),
        'right_mean': paired['right_value'].mean(),
        'mean_difference_right_minus_left': differences.mean(),
        'ci_95_low': np.quantile(bootstrap_means, 0.025),
        'ci_95_high': np.quantile(bootstrap_means, 0.975),
        'right_improved': int((differences > 1e-12).sum()),
        'tied': int((np.abs(differences) <= 1e-12).sum()),
        'right_worsened': int((differences < -1e-12).sum()),
        'preliminary_partial_sample': len(differences) < EXPECTED_REVIEWS,
    }


fair_representation_rows = []
for label, left_metric, right_metric in (
    (
        'Evidence representation',
        'counterfactual_evidence_context_recall',
        'final_evidence_context_recall',
    ),
    (
        'Grounded combined context',
        'counterfactual_grounded_final_context_recall',
        'grounded_final_context_recall',
    ),
):
    result = cross_configuration_bootstrap(
        eligible_df, 'A3', 'A4', left_metric, right_metric
    )
    if result:
        result['metric'] = label
        fair_representation_rows.append(result)
fair_representation_df = pd.DataFrame(fair_representation_rows)
fair_representation_df.to_csv(
    ANALYSIS_DIR / 'a3_chunk_packed_vs_a4_trial_blocks.csv', index=False
)

display(counterfactual_summary_df.round(4))
display(counterfactual_bootstrap_df.round(4))
display(fair_representation_df.round(4))

## 7. Preliminary figures

In [ ]:
def save_figure(name):
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / f'{name}.png', dpi=200, bbox_inches='tight')
    plt.savefig(ANALYSIS_DIR / f'{name}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()


CONFIG_COLOURS = {
    'C1': '#B7BEC7',
    'C2': '#929CAA',
    'C3': '#707C8C',
    'C4': '#4B5563',
    'A2': '#E69F00',
    'A3': '#0072B2',
    'A4': '#009E73',
    'T1': '#66C2A5',
}
METRIC_COLOURS = {
    'evidence': '#0072B2',
    'learnings': '#56B4E9',
    'combined': '#E69F00',
}

if not summary_complete_df.empty:
    plot_summary = summary_complete_df.set_index('run_id').loc[complete_run_ids]

    stage_columns = [
        'macro_retrieval_recall',
        'macro_representation_input_recall',
        'macro_branch_context_recall',
        'macro_final_evidence_context_recall',
    ]
    stage_labels = ['Retrieved', 'Representation input', 'Branch context', 'Final evidence']
    plt.figure(figsize=(10, 5.8))
    for run_id, row in plot_summary.iterrows():
        plt.plot(
            stage_labels, row[stage_columns], marker='o', linewidth=2,
            color=CONFIG_COLOURS.get(run_id, '#4B5563'), label=run_id
        )
    plt.ylabel('Macro ground-truth recall')
    plt.ylim(0, 1.02)
    plt.xticks(rotation=15, ha='right')
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Ground-truth recall through the context-construction funnel')
    plt.legend(ncol=2)
    save_figure('context_recall_funnel_complete_runs')

    plt.figure(figsize=(9, 5))
    bar_colours = [CONFIG_COLOURS.get(run_id, '#4B5563') for run_id in plot_summary.index]
    bars = plt.bar(
        plot_summary.index, plot_summary['macro_grounded_final_context_recall'],
        color=bar_colours
    )
    plt.bar_label(bars, fmt='%.3f', padding=3)
    plt.ylabel('Macro grounded combined-context recall')
    plt.ylim(0, 1.02)
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Ground-truth coverage in the final combined context')
    save_figure('final_context_recall_complete_runs')

    component_columns = [
        'macro_final_evidence_context_recall',
        'macro_grounded_final_learning_context_recall',
        'macro_grounded_final_context_recall',
    ]
    component_labels = ['Final evidence', 'Grounded learnings', 'Grounded combined context']
    component_colours = [
        METRIC_COLOURS['evidence'],
        METRIC_COLOURS['learnings'],
        METRIC_COLOURS['combined'],
    ]
    x = np.arange(len(plot_summary))
    width = 0.24
    plt.figure(figsize=(11, 5.8))
    for offset, (column, label, colour) in enumerate(
        zip(component_columns, component_labels, component_colours)
    ):
        bars = plt.bar(
            x + (offset - 1) * width, plot_summary[column], width,
            label=label, color=colour
        )
        plt.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)
    plt.xticks(x, plot_summary.index)
    plt.ylabel('Macro ground-truth recall')
    plt.ylim(0, 1.08)
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Contribution of evidence representations and generated learnings')
    plt.legend()
    save_figure('final_context_component_recall_complete_runs')

    plt.figure(figsize=(8.5, 5.5))
    for run_id, row in plot_summary.iterrows():
        plt.scatter(
            row['mean_final_words'], row['macro_grounded_final_context_recall'],
            s=90, color=CONFIG_COLOURS.get(run_id, '#4B5563')
        )
        plt.annotate(
            run_id,
            (row['mean_final_words'], row['macro_grounded_final_context_recall']),
            xytext=(6, 6), textcoords='offset points'
        )
    plt.xlabel('Mean final-context words')
    plt.ylabel('Macro grounded combined-context recall')
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25, linestyle='--')
    plt.title('Final-context coverage and context length')
    save_figure('final_context_recall_vs_words')

compression_ids = [
    run_id for run_id in ['C1', 'C2', 'C3', 'C4']
    if run_id in complete_run_ids and run_id in set(counterfactual_summary_df['run_id'])
]
if compression_ids:
    chunk_plot = counterfactual_summary_df.set_index('run_id').loc[compression_ids]
    x = np.arange(len(chunk_plot))
    width = 0.36
    plt.figure(figsize=(9.5, 5.5))
    actual_bars = plt.bar(
        x - width / 2, chunk_plot['actual_evidence_macro_recall'], width,
        label='Observed whole-branch packing', color='#6B7280'
    )
    repacked_bars = plt.bar(
        x + width / 2, chunk_plot['chunk_packed_evidence_macro_recall'], width,
        label='Counterfactual chunk-level packing', color='#A8DADC',
        edgecolor='#0072B2', linewidth=0.9, hatch='//'
    )
    plt.bar_label(actual_bars, fmt='%.3f', padding=3, fontsize=8)
    plt.bar_label(repacked_bars, fmt='%.3f', padding=3, fontsize=8)
    plt.xticks(x, chunk_plot.index)
    plt.ylabel('Macro final evidence recall')
    plt.ylim(0, 1.02)
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Effect of the final packing unit on compressed evidence')
    plt.legend()
    save_figure('counterfactual_chunk_repacking_recall_complete_runs')

if {'A3', 'A4'}.issubset(set(complete_run_ids)) and 'A3' in set(counterfactual_summary_df['run_id']):
    a3_row = counterfactual_summary_df.set_index('run_id').loc['A3']
    a4_row = summary_complete_df.set_index('run_id').loc['A4']
    labels = ['A3 actual\nbranch packing', 'A3 counterfactual\nchunk packing', 'A4 trial blocks']
    evidence_values = [
        a3_row['actual_evidence_macro_recall'],
        a3_row['chunk_packed_evidence_macro_recall'],
        a4_row['macro_final_evidence_context_recall'],
    ]
    combined_values = [
        a3_row['actual_grounded_combined_macro_recall'],
        a3_row['chunk_packed_grounded_combined_macro_recall'],
        a4_row['macro_grounded_final_context_recall'],
    ]
    x = np.arange(len(labels))
    width = 0.36
    plt.figure(figsize=(9.5, 5.7))
    evidence_bars = plt.bar(
        x - width / 2, evidence_values, width, label='Final evidence',
        color=METRIC_COLOURS['evidence']
    )
    combined_bars = plt.bar(
        x + width / 2, combined_values, width, label='Grounded combined context',
        color=METRIC_COLOURS['combined']
    )
    plt.bar_label(evidence_bars, fmt='%.3f', padding=3, fontsize=8)
    plt.bar_label(combined_bars, fmt='%.3f', padding=3, fontsize=8)
    plt.xticks(x, labels)
    plt.ylabel('Macro ground-truth recall')
    plt.ylim(0, 1.02)
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Compression and trial blocks under comparable final packing')
    plt.legend()
    save_figure('a3_chunk_packed_vs_a4_trial_blocks')

# BEGIN RERANKER ORDERING FIGURE
if not reranker_ordering_summary_df.empty:
    reranker_plot = reranker_ordering_summary_df.set_index('K per branch')
    plt.figure(figsize=(8.8, 5.4))
    plt.plot(
        reranker_plot.index,
        reranker_plot['Retrieval-order macro recall'],
        marker='o', linewidth=2.2, color='#6B7280',
        label='Before LLM reranking',
    )
    plt.plot(
        reranker_plot.index,
        reranker_plot['LLM-reranked macro recall'],
        marker='o', linewidth=2.5, color='#0072B2',
        label='After LLM reranking',
    )
    plt.axvline(30, color='#E69F00', linestyle='--', linewidth=1.4, alpha=0.8)
    plt.text(31.5, 0.68, 'Trial-block cut-off', color='#A65F00', fontsize=9)
    plt.xlabel('Candidates retained per branch')
    plt.ylabel('Macro ground-truth recall')
    plt.ylim(0.65, 0.93)
    plt.xticks(RERANKER_K_VALUES)
    plt.grid(axis='y', alpha=0.25, linestyle='--')
    plt.title('Effect of LLM reranking within fixed A3 candidate pools')
    plt.legend()
    save_figure('a3_reranker_ordering_recall_at_k')
# END RERANKER ORDERING FIGURE


In [ ]:
# Configurations central to the context-construction analysis.
requested_runs = ['C4', 'A2', 'A3', 'A4']
available_runs = set(summary_complete_df['run_id'])
selected_runs = [run_id for run_id in requested_runs if run_id in available_runs]

missing_runs = [run_id for run_id in requested_runs if run_id not in available_runs]
if missing_runs:
    print(f"Not yet available as complete runs: {missing_runs}")

selected_summary = (
    summary_complete_df
    .set_index('run_id')
    .loc[selected_runs]
)

In [ ]:
stage_columns = [
    'macro_retrieval_recall',
    'macro_representation_input_recall',
    'macro_branch_context_recall',
    'macro_final_evidence_context_recall',
]

stage_labels = [
    'Retrieved',
    'Representation\ninput',
    'Branch\nevidence',
    'Final\nevidence',
]

selected_colours = {
    'C4': '#4B5563',
    'A2': '#E69F00',
    'A3': '#0072B2',
    'A4': '#009E73',
}

plt.figure(figsize=(9.5, 5.7))

for run_id, row in selected_summary.iterrows():
    plt.plot(
        stage_labels,
        row[stage_columns].to_numpy(dtype=float),
        marker='o',
        markersize=6,
        linewidth=2.4 if run_id != 'A4' else 3.0,
        color=selected_colours.get(run_id, '#4B5563'),
        label=f'{run_id} (selected)' if run_id == 'A4' else run_id,
    )

plt.ylabel('Macro ground-truth recall')
plt.ylim(0, 1.02)
plt.grid(axis='y', alpha=0.25, linestyle='--')
plt.title('Ground-truth retention across evidence-construction stages')
plt.legend(title='Configuration')
save_figure('context_evidence_recall_funnel_selected')

In [ ]:
component_columns = [
    'macro_final_evidence_context_recall',
    'macro_grounded_final_learning_context_recall',
    'macro_grounded_final_context_recall',
]

component_labels = [
    'Final evidence',
    'Grounded learnings',
    'Grounded combined context',
]

component_colours = ['#0072B2', '#56B4E9', '#E69F00']

x = np.arange(len(selected_summary))
width = 0.24

plt.figure(figsize=(10, 5.8))

for index, (column, label, colour) in enumerate(
    zip(component_columns, component_labels, component_colours)
):
    positions = x + (index - 1) * width
    bars = plt.bar(
        positions,
        selected_summary[column],
        width,
        label=label,
        color=colour,
    )
    plt.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)

plt.xticks(x, selected_summary.index)
plt.ylabel('Macro ground-truth recall')
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.25, linestyle='--')
plt.title('Contribution of evidence representations and generated learnings')
plt.legend()
save_figure('final_context_component_recall_selected')

## 8. How to read the preliminary output

For the professor-facing preview, start with `summary_complete_common_reviews.csv` and the component-recall figure. Use retrieval recall only as an input-control metric. Evaluate the representation itself using final evidence recall, evaluate generated synthesis using grounded final learning recall, and use grounded combined recall for evidence coverage in the complete report input. Raw combined final-context recall remains available to describe the text actually supplied to the report-writing model.

A3 and A4 are the principal representation comparison because both configurations use NCT consolidation and LLM reranking. Their actual outputs describe the deployed pipelines, while the counterfactual chunk-packed A3 versus A4 comparison more directly isolates the final evidence representation by removing the branch-level packing disadvantage. C1 is a generic compression baseline, not a representation-only comparator. Do not interpret C5 or T2 as final while `preliminary_partial_sample` is true, and do not compare A2 until its output exists.

A selected chunk is associated with an NCT through the saved web-context metadata. The component parser uses the final packer's fixed ordering, with learnings before evidence, and validates that their union reproduces the post-trim audit. `mean_learning_citation_availability` checks whether NCT IDs cited by retained learnings occurred in the selected branch evidence, although it cannot reconstruct the exact originating branch. Non-ground-truth trials are not labelled irrelevant, so the notebook reports ground-truth coverage and density rather than conventional precision.

The chunk-level repacking results are diagnostic rather than a new experimental configuration. A large improvement over the actual final evidence indicates that branch strings were too coarse for the global packer, but it does not establish report quality and should not be presented as the output of a rerun system.